# River-Flood Forest Restoration NDVI Recovery After Hurricane Melissa

This notebook assesses NDVI recovery trajectories for river-flood forest restoration pixels that provide positive avoided EAD. It compares the pre-event baseline with months 1-2, months 3-4, and months 5-6 after Hurricane Melissa. Damage is defined as a relative NDVI decline of at least 10% in months 1-2 compared with the pre-event baseline, evaluated only where baseline NDVI is at least 0.20.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from IPython.display import display
from rasterio.warp import Resampling, reproject

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)
plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 8,
        "axes.titlesize": 9,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "axes.linewidth": 0.6,
    }
)

## Paths And Constants

In [ ]:
BASE = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
PAPER2 = BASE / "dphil_paper_2"
PAPER3 = BASE / "dphil_paper_3"

OUT_DIR = PAPER3 / "results" / "threats" / "hurricane_melissa_damage" / "recovery" / "river_flood_forest_restoration_ndvi_recovery_months1_6"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RIVER_EAD_MIN_PATH = PAPER2 / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_min.tif"
RIVER_EAD_MAX_PATH = PAPER2 / "processed_data" / "nbs_river_catchment" / "damage_reduction" / "damage_reduction_max.tif"
NDVI_WINDOW_PATHS = {
    "before": PAPER3 / "inputs" / "ndvi" / "HLS_masked_NDVI_2months_before_epsg3448_2025-08-21_to_2025-10-21.tif",
    "months_1_2": PAPER3 / "inputs" / "ndvi" / "HLS_masked_NDVI_2months_after_epsg3448_2025-10-29_to_2025-12-29.tif",
    "months_3_4": PAPER3 / "inputs" / "ndvi" / "HLS_masked_NDVI_months3to4_after_epsg3448_2025-12-30_to_2026-02-28.tif",
    "months_5_6": PAPER3 / "inputs" / "ndvi" / "HLS_masked_NDVI_months5to6_after_epsg3448_2026-03-01_to_2026-04-29.tif",
}
NDVI_WINDOW_LABELS = {
    "before": "Pre-event",
    "months_1_2": "Months 1-2",
    "months_3_4": "Months 3-4",
    "months_5_6": "Months 5-6",
}
NDVI_TIME_ORDER = ["before", "months_1_2", "months_3_4", "months_5_6"]

J2USD = 1.0 / 150.0
REL_BASELINE_MIN = 0.20
REL_DAMAGE_THRESHOLD = -0.10
FIGURE_DPI = 300

for input_path in [RIVER_EAD_MIN_PATH, RIVER_EAD_MAX_PATH, *NDVI_WINDOW_PATHS.values()]:
    if not input_path.exists():
        raise FileNotFoundError(input_path)

OUT_DIR

## Helper Functions

In [ ]:
def read_positive_ead_usd(path: Path, reference_profile: dict | None = None) -> tuple[np.ndarray, dict]:
    """Read avoided EAD raster, convert JMD to USD, and retain only positive pixels."""
    with rasterio.open(path) as source_raster:
        profile = source_raster.profile.copy()
        ead_array = source_raster.read(1).astype("float64") * J2USD
    ead_array[~np.isfinite(ead_array) | (ead_array <= 0)] = np.nan
    if reference_profile is not None:
        alignment_checks = {
            "crs": profile["crs"] == reference_profile["crs"],
            "transform": profile["transform"] == reference_profile["transform"],
            "height": profile["height"] == reference_profile["height"],
            "width": profile["width"] == reference_profile["width"],
        }
        if not all(alignment_checks.values()):
            raise ValueError(f"Raster alignment mismatch for {path}: {alignment_checks}")
    return ead_array, profile


def reproject_ndvi_to_reference(path: Path, reference_profile: dict) -> np.ndarray:
    """Reproject a continuous NDVI raster to the river restoration-benefit grid."""
    destination = np.full(
        (reference_profile["height"], reference_profile["width"]),
        np.nan,
        dtype="float32",
    )
    with rasterio.open(path) as source_raster:
        reproject(
            source=rasterio.band(source_raster, 1),
            destination=destination,
            src_transform=source_raster.transform,
            src_crs=source_raster.crs,
            src_nodata=source_raster.nodata,
            dst_transform=reference_profile["transform"],
            dst_crs=reference_profile["crs"],
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )
    return destination


def valid_ndvi(ndvi_array: np.ndarray) -> np.ndarray:
    """Return valid NDVI values on the expected -1 to 1 range."""
    return np.isfinite(ndvi_array) & (ndvi_array >= -1.0) & (ndvi_array <= 1.0)


def pct(numerator: float, denominator: float) -> float:
    """Return percentage, preserving NaN when the denominator is zero."""
    return float(numerator / denominator * 100) if denominator else np.nan


def safe_mean(values: np.ndarray) -> float:
    """Return a finite mean or NaN for empty arrays."""
    return float(np.nanmean(values)) if values.size else np.nan


def ead_sum_usd(ead_array: np.ndarray, mask: np.ndarray) -> float:
    """Sum positive avoided EAD values inside a mask."""
    return float(np.nansum(np.where(mask & np.isfinite(ead_array), ead_array, 0.0)))


def save_figure(figure: plt.Figure, stem: str) -> list[Path]:
    """Save a figure to PNG, PDF, and SVG."""
    output_paths = []
    for suffix in ["png", "pdf", "svg"]:
        output_path = OUT_DIR / f"{stem}.{suffix}"
        figure.savefig(output_path, dpi=FIGURE_DPI, bbox_inches="tight", facecolor="white")
        output_paths.append(output_path)
    return output_paths


def summarize_recovery_group(
    group_name: str,
    group_mask: np.ndarray,
    ndvi_arrays: dict[str, np.ndarray],
    river_ead_min_usd: np.ndarray,
    river_ead_max_usd: np.ndarray,
    total_ead_min_usd: float,
    total_ead_max_usd: float,
    pixel_area_ha: float,
) -> dict:
    """Summarize NDVI change, recovery, and avoided EAD for one pixel group."""
    before_values = ndvi_arrays["before"][group_mask]
    months_1_2_values = ndvi_arrays["months_1_2"][group_mask]
    months_3_4_values = ndvi_arrays["months_3_4"][group_mask]
    months_5_6_values = ndvi_arrays["months_5_6"][group_mask]
    initial_drop = before_values - months_1_2_values
    positive_drop = initial_drop > 0

    recovery_fraction_3_4 = np.full(before_values.shape, np.nan, dtype="float64")
    recovery_fraction_5_6 = np.full(before_values.shape, np.nan, dtype="float64")
    recovery_fraction_3_4[positive_drop] = (months_3_4_values[positive_drop] - months_1_2_values[positive_drop]) / initial_drop[positive_drop]
    recovery_fraction_5_6[positive_drop] = (months_5_6_values[positive_drop] - months_1_2_values[positive_drop]) / initial_drop[positive_drop]

    pixel_count = int(group_mask.sum())
    avoided_ead_min_usd = ead_sum_usd(river_ead_min_usd, group_mask)
    avoided_ead_max_usd = ead_sum_usd(river_ead_max_usd, group_mask)
    return {
        "group": group_name,
        "pixel_count": pixel_count,
        "area_ha": pixel_count * pixel_area_ha,
        "positive_avoided_ead_usd_minimum": avoided_ead_min_usd,
        "positive_avoided_ead_usd_maximum": avoided_ead_max_usd,
        "pct_total_positive_avoided_ead_minimum": pct(avoided_ead_min_usd, total_ead_min_usd),
        "pct_total_positive_avoided_ead_maximum": pct(avoided_ead_max_usd, total_ead_max_usd),
        "mean_ndvi_before": safe_mean(before_values),
        "mean_ndvi_months_1_2": safe_mean(months_1_2_values),
        "mean_ndvi_months_3_4": safe_mean(months_3_4_values),
        "mean_ndvi_months_5_6": safe_mean(months_5_6_values),
        "mean_delta_months_1_2_minus_before": safe_mean(months_1_2_values - before_values),
        "mean_delta_months_3_4_minus_months_1_2": safe_mean(months_3_4_values - months_1_2_values),
        "mean_delta_months_5_6_minus_months_3_4": safe_mean(months_5_6_values - months_3_4_values),
        "mean_delta_months_5_6_minus_before": safe_mean(months_5_6_values - before_values),
        "mean_relative_change_months_1_2_pct": safe_mean((months_1_2_values - before_values) / before_values * 100),
        "mean_relative_change_months_3_4_pct": safe_mean((months_3_4_values - before_values) / before_values * 100),
        "mean_relative_change_months_5_6_pct": safe_mean((months_5_6_values - before_values) / before_values * 100),
        "pct_improved_months_3_4_vs_months_1_2": pct(np.sum(months_3_4_values > months_1_2_values), pixel_count),
        "pct_improved_months_5_6_vs_months_3_4": pct(np.sum(months_5_6_values > months_3_4_values), pixel_count),
        "pct_improved_months_5_6_vs_months_1_2": pct(np.sum(months_5_6_values > months_1_2_values), pixel_count),
        "pct_recovered_to_before_by_months_3_4": pct(np.sum(months_3_4_values >= before_values), pixel_count),
        "pct_recovered_to_before_by_months_5_6": pct(np.sum(months_5_6_values >= before_values), pixel_count),
        "mean_recovery_fraction_of_initial_drop_by_months_3_4_pct": safe_mean(recovery_fraction_3_4[np.isfinite(recovery_fraction_3_4)] * 100),
        "mean_recovery_fraction_of_initial_drop_by_months_5_6_pct": safe_mean(recovery_fraction_5_6[np.isfinite(recovery_fraction_5_6)] * 100),
        "mean_recovery_rate_ndvi_per_month_months_3_4_vs_months_1_2": safe_mean((months_3_4_values - months_1_2_values) / 2),
        "mean_recovery_rate_ndvi_per_month_months_5_6_vs_months_3_4": safe_mean((months_5_6_values - months_3_4_values) / 2),
        "mean_recovery_fraction_of_initial_drop_per_month_by_months_3_4_pct": safe_mean(recovery_fraction_3_4[np.isfinite(recovery_fraction_3_4)] * 100 / 2),
        "mean_recovery_fraction_of_initial_drop_per_month_by_months_5_6_pct": safe_mean(recovery_fraction_5_6[np.isfinite(recovery_fraction_5_6)] * 100 / 4),
    }

## Load River-Flood Benefit Pixels And NDVI Windows

In [ ]:
river_ead_min_usd, river_profile = read_positive_ead_usd(RIVER_EAD_MIN_PATH)
river_ead_max_usd, _ = read_positive_ead_usd(RIVER_EAD_MAX_PATH, river_profile)
river_transform = river_profile["transform"]
river_pixel_area_ha = abs(river_transform.a * river_transform.e) / 10_000
river_benefit_mask = np.isfinite(river_ead_min_usd) | np.isfinite(river_ead_max_usd)

ndvi_arrays = {
    window_name: reproject_ndvi_to_reference(window_path, river_profile)
    for window_name, window_path in NDVI_WINDOW_PATHS.items()
}

total_ead_min_usd = ead_sum_usd(river_ead_min_usd, np.isfinite(river_ead_min_usd))
total_ead_max_usd = ead_sum_usd(river_ead_max_usd, np.isfinite(river_ead_max_usd))

print(f"River-flood restoration benefit area: {river_benefit_mask.sum() * river_pixel_area_ha:,.1f} ha")
print(f"Positive avoided EAD: US${total_ead_min_usd / 1e6:,.2f}-{total_ead_max_usd / 1e6:,.2f} million")
print(f"Pixel area: {river_pixel_area_ha:.3f} ha")

## Classify Month 1-2 Damage And Full Recovery Series

In [ ]:
valid_masks = {window_name: valid_ndvi(ndvi_array) for window_name, ndvi_array in ndvi_arrays.items()}
baseline_eligible_mask = river_benefit_mask & valid_masks["before"] & (ndvi_arrays["before"] >= REL_BASELINE_MIN)
full_series_mask = baseline_eligible_mask.copy()
for window_name in ["months_1_2", "months_3_4", "months_5_6"]:
    full_series_mask &= valid_masks[window_name]

months_1_2_valid_mask = baseline_eligible_mask & valid_masks["months_1_2"]
relative_change_months_1_2 = np.full(river_benefit_mask.shape, np.nan, dtype="float32")
np.divide(
    ndvi_arrays["months_1_2"] - ndvi_arrays["before"],
    ndvi_arrays["before"],
    out=relative_change_months_1_2,
    where=months_1_2_valid_mask,
)
damaged_months_1_2_mask = months_1_2_valid_mask & (relative_change_months_1_2 <= REL_DAMAGE_THRESHOLD)
damaged_full_series_mask = damaged_months_1_2_mask & valid_masks["months_3_4"] & valid_masks["months_5_6"]

coverage_summary = pd.DataFrame(
    [
        {"metric": "river_benefit_area_ha", "value": river_benefit_mask.sum() * river_pixel_area_ha},
        {"metric": "baseline_eligible_area_ha", "value": baseline_eligible_mask.sum() * river_pixel_area_ha},
        {"metric": "full_series_area_ha", "value": full_series_mask.sum() * river_pixel_area_ha},
        {"metric": "damaged_months_1_2_area_ha", "value": damaged_months_1_2_mask.sum() * river_pixel_area_ha},
        {"metric": "damaged_months_1_2_full_series_area_ha", "value": damaged_full_series_mask.sum() * river_pixel_area_ha},
        {"metric": "pct_full_series_area_damaged_months_1_2", "value": pct(damaged_full_series_mask.sum(), full_series_mask.sum())},
        {"metric": "positive_avoided_ead_usd_minimum_total", "value": total_ead_min_usd},
        {"metric": "positive_avoided_ead_usd_maximum_total", "value": total_ead_max_usd},
    ]
)
coverage_summary

## Pixel-Level Recovery Summaries

In [ ]:
pixel_recovery_summary = pd.DataFrame(
    [
        summarize_recovery_group(
            "all_river_flood_restoration_benefit_pixels_with_full_series",
            full_series_mask,
            ndvi_arrays,
            river_ead_min_usd,
            river_ead_max_usd,
            total_ead_min_usd,
            total_ead_max_usd,
            river_pixel_area_ha,
        ),
        summarize_recovery_group(
            "damaged_months_1_2_river_flood_restoration_benefit_pixels_with_full_series",
            damaged_full_series_mask,
            ndvi_arrays,
            river_ead_min_usd,
            river_ead_max_usd,
            total_ead_min_usd,
            total_ead_max_usd,
            river_pixel_area_ha,
        ),
    ]
)
pixel_recovery_summary

## Time-Window Summaries

In [ ]:
time_window_rows = []
for group_name, group_mask in [
    ("all_river_flood_restoration_benefit_pixels_with_full_series", full_series_mask),
    ("damaged_months_1_2_river_flood_restoration_benefit_pixels_with_full_series", damaged_full_series_mask),
]:
    before_values = ndvi_arrays["before"][group_mask]
    for window_name in NDVI_TIME_ORDER:
        window_values = ndvi_arrays[window_name][group_mask]
        time_window_rows.append(
            {
                "group": group_name,
                "window": window_name,
                "window_label": NDVI_WINDOW_LABELS[window_name],
                "pixel_count": int(group_mask.sum()),
                "area_ha": group_mask.sum() * river_pixel_area_ha,
                "positive_avoided_ead_usd_minimum": ead_sum_usd(river_ead_min_usd, group_mask),
                "positive_avoided_ead_usd_maximum": ead_sum_usd(river_ead_max_usd, group_mask),
                "mean_ndvi": safe_mean(window_values),
                "mean_delta_vs_before": safe_mean(window_values - before_values),
                "mean_relative_change_vs_before_pct": safe_mean((window_values - before_values) / before_values * 100),
            }
        )
time_window_summary = pd.DataFrame(time_window_rows)
time_window_summary

## Figures

In [ ]:
fig, axis = plt.subplots(figsize=(92 / 25.4, 70 / 25.4), dpi=FIGURE_DPI)
legend_labels = {
    "all_river_flood_restoration_benefit_pixels_with_full_series": "All benefit pixels",
    "damaged_months_1_2_river_flood_restoration_benefit_pixels_with_full_series": "Damaged in months 1-2",
}
for group_name, color in [
    ("all_river_flood_restoration_benefit_pixels_with_full_series", "#238b45"),
    ("damaged_months_1_2_river_flood_restoration_benefit_pixels_with_full_series", "#d73027"),
]:
    plot_rows = time_window_summary[time_window_summary["group"].eq(group_name)].set_index("window").loc[NDVI_TIME_ORDER]
    axis.plot(
        plot_rows["window_label"],
        plot_rows["mean_ndvi"],
        marker="o",
        linewidth=1.2,
        color=color,
        label=legend_labels[group_name],
    )
axis.axhline(REL_BASELINE_MIN, color="#7f7f7f", linewidth=0.7, linestyle="--", label="Baseline eligibility threshold")
axis.set_ylabel("Mean NDVI")
axis.set_title("River-flood restoration benefit pixel NDVI trajectory")
axis.grid(axis="y", linewidth=0.35, alpha=0.35)
axis.spines[["top", "right"]].set_visible(False)
axis.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.24), ncol=2)
fig.tight_layout()
mean_ndvi_figure_paths = save_figure(fig, "river_flood_forest_restoration_mean_ndvi_recovery_months1_6")
display(fig)
plt.close(fig)

fig, axis = plt.subplots(figsize=(92 / 25.4, 70 / 25.4), dpi=FIGURE_DPI)
damaged_row = pixel_recovery_summary[
    pixel_recovery_summary["group"].eq("damaged_months_1_2_river_flood_restoration_benefit_pixels_with_full_series")
].iloc[0]
bar_labels = ["Recovered by\nmonths 3-4", "Recovered by\nmonths 5-6", "Improved\n3-4 vs 1-2", "Improved\n5-6 vs 3-4"]
bar_values = [
    damaged_row["pct_recovered_to_before_by_months_3_4"],
    damaged_row["pct_recovered_to_before_by_months_5_6"],
    damaged_row["pct_improved_months_3_4_vs_months_1_2"],
    damaged_row["pct_improved_months_5_6_vs_months_3_4"],
]
axis.bar(bar_labels, bar_values, color=["#1a9850", "#006837", "#74add1", "#4575b4"], width=0.62)
axis.set_ylim(0, 100)
axis.set_ylabel("Damaged benefit pixels (%)")
axis.set_title("Recovery indicators for restoration pixels damaged in months 1-2")
axis.grid(axis="y", linewidth=0.35, alpha=0.35)
axis.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
recovery_indicator_figure_paths = save_figure(fig, "river_flood_forest_restoration_recovery_indicators_months1_6")
display(fig)
plt.close(fig)

mean_ndvi_figure_paths + recovery_indicator_figure_paths

## Export Tables And Metadata

In [ ]:
coverage_summary_path = OUT_DIR / "river_flood_forest_restoration_recovery_coverage_summary.csv"
pixel_recovery_summary_path = OUT_DIR / "river_flood_forest_restoration_recovery_pixel_summary.csv"
time_window_summary_path = OUT_DIR / "river_flood_forest_restoration_recovery_time_window_summary.csv"
metadata_path = OUT_DIR / "river_flood_forest_restoration_recovery_metadata.csv"

metadata = pd.DataFrame(
    [
        {"name": "river_ead_min_path", "value": str(RIVER_EAD_MIN_PATH)},
        {"name": "river_ead_max_path", "value": str(RIVER_EAD_MAX_PATH)},
        {"name": "ndvi_before_path", "value": str(NDVI_WINDOW_PATHS["before"])},
        {"name": "ndvi_months_1_2_path", "value": str(NDVI_WINDOW_PATHS["months_1_2"])},
        {"name": "ndvi_months_3_4_path", "value": str(NDVI_WINDOW_PATHS["months_3_4"])},
        {"name": "ndvi_months_5_6_path", "value": str(NDVI_WINDOW_PATHS["months_5_6"])},
        {"name": "baseline_eligibility_threshold", "value": str(REL_BASELINE_MIN)},
        {"name": "damage_threshold", "value": "relative NDVI change months 1-2 vs before <= -0.10"},
        {"name": "currency_conversion", "value": "JMD to USD = 1/150 for river-flood avoided EAD rasters"},
        {"name": "pixel_area_ha", "value": str(river_pixel_area_ha)},
        {"name": "benefit_area_definition", "value": "pixels with positive avoided EAD in either minimum or maximum river-flood forest restoration scenario"},
    ]
)

coverage_summary.to_csv(coverage_summary_path, index=False)
pixel_recovery_summary.to_csv(pixel_recovery_summary_path, index=False)
time_window_summary.to_csv(time_window_summary_path, index=False)
metadata.to_csv(metadata_path, index=False)

[
    coverage_summary_path,
    pixel_recovery_summary_path,
    time_window_summary_path,
    metadata_path,
]

## Key Results

In [ ]:
display(coverage_summary)
display(pixel_recovery_summary)
display(time_window_summary)